# MAIA — QLoRA run `exp_v1_r16lr2e4e3`


Generated by `maia.training.colab`. **Do not edit the hyperparameters here** —
they live in `configs/train/full-r16.yaml` and this notebook reads them, so the committed
YAML stays the single source of truth for what ran (D-0025).

Expected VRAM: **14.3 GiB estimated** on `L4` (24 GiB).

Colab cuts sessions off, so **re-running this notebook resumes** from the newest
checkpoint rather than starting over.

In [ ]:
# GPU actually allocated — Colab does not always give what was requested.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Unsloth pulls its own pinned torch/trl/peft set; installing them separately is how
# a Colab environment ends up with a mismatched stack.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
!git clone --depth 1 --branch main https://github.com/ericrisco/maia /content/maia
!pip install -q -e /content/maia
import os

os.chdir("/content/maia")

In [ ]:
from maia.training.colab import resume_from, usable, vram_estimate
from maia.training.config import load_config, render

config = load_config("configs/train/full-r16.yaml")
print(render(config))
assert usable(config, "L4"), (
    f"{vram_estimate(config):.1f} GiB estimated does not fit L4 — halve "
    "per_device_batch_size and double gradient_accumulation_steps to keep the same "
    "effective batch"
)

In [ ]:
from maia.synth.publish import read_dataset
from maia.training.chat import format_dataset

examples = read_dataset("data/dataset.jsonl")
train = [e for e in examples if e.split.value == "train"]
rows = format_dataset(train)
print(f"{len(rows)} training row(s)")

In [ ]:
import wandb
from google.colab import userdata

# Colab secrets, not literals: a token pasted into a cell is a token in the notebook.
wandb.login(key=userdata.get("WANDB_API_KEY"))

In [ ]:
from maia.training.unsloth_runner import unsloth_trainer

trainer = unsloth_trainer()
job = trainer.build(config, rows)
checkpoint = resume_from(config)
print(f"resuming from {checkpoint}" if checkpoint else "starting fresh")
job.train(resume_from_checkpoint=str(checkpoint) if checkpoint else None)

In [ ]:
from maia.training.smoke import TrainingOutcome, check_loss
from maia.training.unsloth_runner import epoch_checkpoints, losses_from_log

losses = losses_from_log(job.state.log_history)
outcome = TrainingOutcome(
    losses=losses,
    checkpoint=epoch_checkpoints(config)[-1],
    hours=job.state.log_history[-1].get("train_runtime", 0) / 3600,
)
print(check_loss(outcome).detail)

In [ ]:
# Checkpoints outlive the runtime only if they leave it.
from google.colab import userdata
from huggingface_hub import HfApi

HfApi(token=userdata.get("HF_TOKEN")).upload_folder(
    repo_id=f"ericrisco/maia-12b-{config.name}",
    folder_path=str(config.output_dir / config.name),
    repo_type="model",
    commit_message=f"{config.name}: adapters",
)